In [2]:
from scipy import sparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import os
import seaborn as sns
import time

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap as up

from tools import plot_3D, plot_the_PCA_interval, plot_clusters

os.environ["MKL_THREADING_LAYER"] = "GNU"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
RANDOM_SEED = 42

In [3]:
X = sparse.load_npz("../datasets/clusterization/train.npz")

In [4]:
print(f"Размер: {X.shape}")
M, N = X.shape
print(X[:1, :].toarray().max(), X[:1, :].toarray().min(), X[:1, :].toarray().mean())

Размер: (21000, 3049)
7598.588105474598 -13.815510557964274 2.568935238165224


In [5]:
all_lens = []
for i in range(M):
    row = X[i].data
    all_lens.append(len(row.tolist()))
arr = np.array(all_lens)
values, counts = np.unique(arr, return_counts=True) 
print(f"Мода по размерности: {values[np.argmax(counts)]}")

Мода по размерности: 554


### <div align="center">PCA</div>

In [6]:
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

In [7]:
scaler = RobustScaler(with_centering=False)
X_scaled = scaler.fit_transform(X)

In [8]:
# pca = PCA()
# pca.fit(X_scaled)

In [9]:
# start, stop, step = 0, 50, 1
# plot_the_PCA_interval(pca, start, stop, step, show_ratios=True, print_cum=True)

### Conclusion (PCA):
Можно просто урезать размерность до 40-50 компонент не теряя дисперсии, попробуем это применить перед t-sne и umap

In [10]:
pca_new = PCA(n_components=40)
X_embedded_pca = pca_new.fit_transform(X_scaled)

### <div align="center">t-SNE</div>

In [11]:
# from sklearn.manifold import TSNE

In [12]:
# X_embedded_pca_tsne = TSNE(n_components=3, 
#                   init='random').fit_transform(X_embedded_pca)
# plot_3D(X_embedded_pca_tsne, title="PCA + t-SNE")

### <div align=center> UMAP </div>

In [13]:
import umap as up

In [15]:
neigh = 15
umap = up.UMAP(n_components=2, n_neighbors=neigh)
X_embedded_pca_umap = umap.fit_transform(X_embedded_pca)
plot_3D(X_embedded_pca_umap, title=f"PCA+UMAP_n{neigh}")

### <div align="center">Clusterization</div>

In [16]:
from sklearn.cluster import HDBSCAN
import time
import seaborn as sns

In [19]:
plot_clusters(X_embedded_pca,
              HDBSCAN, 
              [], 
              {'min_cluster_size': 100, 'metric': 'cosine'}
)

c:\Users\user_1\miniforge3\envs\ipynb_VK\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


silhouette_score: 0.5210725157170277
Данные по классам: {np.int64(-1): np.int64(6110), np.int64(0): np.int64(116), np.int64(1): np.int64(108), np.int64(2): np.int64(104), np.int64(3): np.int64(160), np.int64(4): np.int64(140), np.int64(5): np.int64(482), np.int64(6): np.int64(438), np.int64(7): np.int64(534), np.int64(8): np.int64(526), np.int64(9): np.int64(238), np.int64(10): np.int64(163), np.int64(11): np.int64(165), np.int64(12): np.int64(171), np.int64(13): np.int64(143), np.int64(14): np.int64(249), np.int64(15): np.int64(157), np.int64(16): np.int64(176), np.int64(17): np.int64(192), np.int64(18): np.int64(166), np.int64(19): np.int64(339), np.int64(20): np.int64(171), np.int64(21): np.int64(252), np.int64(22): np.int64(108), np.int64(23): np.int64(351), np.int64(24): np.int64(201), np.int64(25): np.int64(8678), np.int64(26): np.int64(362)}


Clusters found by HDBSCAN
Clustering took 20.44 s


HDBSCAN - очень плохие метрики на тестовых данных

#### k-MEANS

In [22]:
import sklearn.cluster as cluster

plot_clusters(X_embedded_pca, cluster.KMeans, (), {'n_clusters':150})

silhouette_score: 0.7056206351637985
Данные по классам: {np.int32(0): np.int64(9335), np.int32(1): np.int64(343), np.int32(2): np.int64(211), np.int32(3): np.int64(37), np.int32(4): np.int64(17), np.int32(5): np.int64(19), np.int32(6): np.int64(346), np.int32(7): np.int64(296), np.int32(8): np.int64(548), np.int32(9): np.int64(588), np.int32(10): np.int64(469), np.int32(11): np.int64(510), np.int32(12): np.int64(399), np.int32(13): np.int64(21), np.int32(14): np.int64(28), np.int32(15): np.int64(142), np.int32(16): np.int64(306), np.int32(17): np.int64(277), np.int32(18): np.int64(309), np.int32(19): np.int64(122), np.int32(20): np.int64(33), np.int32(21): np.int64(26), np.int32(22): np.int64(36), np.int32(23): np.int64(267), np.int32(24): np.int64(69), np.int32(25): np.int64(84), np.int32(26): np.int64(23), np.int32(27): np.int64(44), np.int32(28): np.int64(193), np.int32(29): np.int64(170), np.int32(30): np.int64(204), np.int32(31): np.int64(223), np.int32(32): np.int64(28), np.int32

Clusters found by KMeans
Clustering took 7.66 s


k-MEANS - ещё хуже. Но тут можно попробовать сделать перебор кол-ва кластеров, но для этого сначала надо реализовать Silhouette

#### AgglomerativeClustering

In [21]:
from sklearn.cluster import AgglomerativeClustering

plot_clusters(X_embedded_pca, AgglomerativeClustering, (), {'n_clusters': 5, 'linkage': 'average', 'metric': 'cosine'})

silhouette_score: 0.30167433682559014
Данные по классам: {np.int64(0): np.int64(2194), np.int64(1): np.int64(4226), np.int64(2): np.int64(2785), np.int64(3): np.int64(10152), np.int64(4): np.int64(1643)}


Clusters found by AgglomerativeClustering
Clustering took 35.95 s


#### SpectralClustering

In [ ]:
from sklearn.cluster import SpectralClustering
from scipy import sparse
import numpy as np
import pandas as pd


X = sparse.load_npz("../datasets/clusterization/train.npz")
labels = SpectralClustering(n_clusters = 5, n_neighbors = 10).fit_predict(X_embedded_pca)
subm = pd.DataFrame({"ID": np.arange(labels.size), "TARGET": labels})
subm.to_csv(f"subm_{SpectralClustering.__name__}.csv", index=False)


c:\Users\user_1\miniforge3\envs\ipynb_VK\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


### GaussianMixture

In [24]:
from sklearn.mixture import GaussianMixture

plot_clusters(X_embedded_pca, GaussianMixture, (), {'n_components': 8})

silhouette_score: 0.07941109408619436
Данные по классам: {np.int64(0): np.int64(2035), np.int64(1): np.int64(743), np.int64(2): np.int64(9544), np.int64(3): np.int64(2132), np.int64(4): np.int64(730), np.int64(5): np.int64(979), np.int64(6): np.int64(4347), np.int64(7): np.int64(490)}


Clusters found by GaussianMixture
Clustering took 13.37 s
